In [4]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import random

# Input paths
BASE_DIR = Path(r"C:\Users\adrme\OneDrive\Music\Project\DataSynthUse")
IN_DIR_NAT = BASE_DIR / 'Cleaned_Data1'
IN_DIR_ST = BASE_DIR / 'Cleaned_Data2'

# Output paths
OUT_DIR = Path(r"C:\Users\adrme\OneDrive\Music\Project\SynthData")
OUT_DIR_NAT = OUT_DIR / 'Cleaned_Data1'
OUT_DIR_ST = OUT_DIR / 'Cleaned_Data2'

# Synthetic Timeline (3 Years)
START_DATE = datetime(2024, 1, 1)
DAYS_TO_GENERATE = 365 * 3

# Noise parameters (+/- 3% variation)
NOISE_FACTOR = 0.03

def setup_directories():
    print("Setting up output directories...")
    os.makedirs(OUT_DIR_NAT, exist_ok=True)
    os.makedirs(OUT_DIR_ST, exist_ok=True)

def load_historical_data_by_month(directory):
    print(f"Loading historical baselines from {directory}...")
    monthly_data = {i: [] for i in range(1, 13)}
    
    files = list(Path(directory).rglob('*.xlsx'))
    
    valid_files = [f for f in files if not f.name.startswith('~')]
    
    if not valid_files:
        print(f"No valid Excel files found in {directory}")
        return monthly_data

    for file in valid_files:
        try:
            filename = file.stem
            parts = filename.split('-')
            if len(parts) >= 2:
                month = int(parts[1])
            else:
                month = int(file.parent.name)
                
            df = pd.read_excel(file)
            monthly_data[month].append(df)
            
        except Exception as e:
            print(f"Skipping unreadable file {file.name}: {e}")
            continue
            
    return monthly_data

def apply_noise(df, is_percentage=False):
    synthetic_df = df.copy()
    numeric_cols = synthetic_df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        noise = np.random.uniform(-NOISE_FACTOR, NOISE_FACTOR, size=len(synthetic_df))
        synthetic_df[col] = synthetic_df[col] * (1 + noise)
        
        # Convert column name to string safely to avoid TypeError
        col_str = str(col)
        
        if 'Percentage' in col_str or 'Met' in col_str or is_percentage:
            synthetic_df[col] = np.clip(synthetic_df[col], 0, 100)
        elif 'Factor' in col_str:
            synthetic_df[col] = np.clip(synthetic_df[col], 0, 1)
        else:
            synthetic_df[col] = np.clip(synthetic_df[col], 0, None)
            
        synthetic_df[col] = np.round(synthetic_df[col], 2)
        
    return synthetic_df

def save_synthetic_file(df, base_out_dir, current_date):
    year_folder = base_out_dir / str(current_date.year)
    month_folder = year_folder / f"{current_date.month:02d}"
    os.makedirs(month_folder, exist_ok=True)
    
    file_name = current_date.strftime('%d-%m-%Y') + '.xlsx'
    file_path = month_folder / file_name
    
    df.to_excel(file_path, index=False)

def generate_synthetic_data():
    setup_directories()
    
    nat_baselines = load_historical_data_by_month(IN_DIR_NAT)
    st_baselines = load_historical_data_by_month(IN_DIR_ST)
    
    print(f"Generating {DAYS_TO_GENERATE} days of synthetic data...")
    
    for day_offset in range(DAYS_TO_GENERATE):
        current_date = START_DATE + timedelta(days=day_offset)
        target_month = current_date.month
        
        if nat_baselines[target_month]:
            base_df_nat = random.choice(nat_baselines[target_month])
            synth_nat = apply_noise(base_df_nat)
            save_synthetic_file(synth_nat, OUT_DIR_NAT, current_date)
            
        if st_baselines[target_month]:
            base_df_st = random.choice(st_baselines[target_month])
            synth_st = apply_noise(base_df_st)
            save_synthetic_file(synth_st, OUT_DIR_ST, current_date)
            
        if (day_offset + 1) % 100 == 0:
            print(f"Processed {day_offset + 1} / {DAYS_TO_GENERATE} days...")

    print("Generation Complete. Synthetic data saved to:", OUT_DIR)

# Run the generator
generate_synthetic_data()

Setting up output directories...
Loading historical baselines from C:\Users\adrme\OneDrive\Music\Project\DataSynthUse\Cleaned_Data1...
Loading historical baselines from C:\Users\adrme\OneDrive\Music\Project\DataSynthUse\Cleaned_Data2...
Generating 1095 days of synthetic data...
Processed 100 / 1095 days...
Processed 200 / 1095 days...
Processed 300 / 1095 days...
Processed 400 / 1095 days...
Processed 500 / 1095 days...
Processed 600 / 1095 days...
Processed 700 / 1095 days...
Processed 800 / 1095 days...
Processed 900 / 1095 days...
Processed 1000 / 1095 days...
Generation Complete. Synthetic data saved to: C:\Users\adrme\OneDrive\Music\Project\SynthData
